In [1]:
import argparse
import copy
import dill as pickle
import numpy as np
import os
import sys
import time

from aero_optim.mf_sm.mf_models import get_model, get_sampler, MultiObjectiveModel
from aero_optim.utils import check_config, mv_filelist

sys.path.append(os.path.join(os.getcwd(), ".."))
from cascade_mf.main_mf_sm import (set_mo_DOE, save_results, print, # noqa
                                   compute_bayesian_infill, execute_single_gen)

In [2]:
# intit problem
sm_config, _, _ = check_config("cascade_mf.json", optim=True)
# study
pod = sm_config["study"]["ffd_type"] == "ffd_pod_2d"
outdir = sm_config["study"]["outdir"]
# ffd
ffd_config = sm_config["ffd"]
# optim
optim_config = sm_config["optim"]
seed = optim_config.get("seed", 123)

# get bound
n_design = optim_config["n_design"] if not pod else ffd_config["pod_ncontrol"]
bound = sm_config["optim"]["bound"]
bound = ([bound[0]] * n_design, [bound[-1]] * n_design)

# update bound in case of rotation
if ffd_config.get("rotation", False):
    rot_bound = ffd_config["rot_bound"]
    bound = (bound[0] + [rot_bound[0]], bound[-1] + [rot_bound[-1]])
    n_design += 1

AERO-Optim: general check config..


In [3]:
model_name = sm_config["optim"]["model_name"]
# loss model
model_loss_ADP = get_model(model_name, n_design, sm_config, outdir, seed)
model_loss_OP = copy.deepcopy(model_loss_ADP)
model_loss = MultiObjectiveModel([model_loss_ADP, model_loss_OP])
# outflow angle model
model_angle_ADP = copy.deepcopy(model_loss_ADP)
model_angle_OP1 = copy.deepcopy(model_loss_ADP)
model_angle_OP2 = copy.deepcopy(model_loss_ADP)
model_angle = MultiObjectiveModel([model_angle_ADP, model_angle_OP1, model_angle_OP2])

In [4]:
mf_sampler = get_sampler(
    n_design, bound, seed, model_loss.requires_nested_doe
)
x_lf, x_hf = mf_sampler.sample_mf(sm_config["optim"]["n_lf"], sm_config["optim"]["n_hf"])
print(f"DEBUG -- x_lf: {x_lf.shape}, x_hf: {x_hf.shape}")

DEBUG -- x_lf: (99, 5), x_hf: (9, 5)


In [5]:
x_lf_d = np.loadtxt("output_mf/lf_doe/candidates.txt")
x_hf_d = np.loadtxt("output_mf/hf_doe/candidates.txt")

In [6]:
with open(os.path.join("output_mf/lf_doe", "df_dict.pkl"), "rb") as handle:
        lf_dict = pickle.load(handle)
with open(os.path.join("output_mf/hf_doe", "df_dict.pkl"), "rb") as handle:
        hf_dict = pickle.load(handle)

In [ ]:
# compute DOEs
# lf_DOE
# QoIs
QoI = sm_config["optim"]["QoI"]
lf_w_ADP = np.array([lf_dict[0][cid]["ADP"][QoI].dropna().iloc[-1] for cid in range(len(lf_dict[0]))])
lf_w_OP = np.array(
    [0.5 * (lf_dict[0][cid]["OP1"][QoI].dropna().iloc[-1] + lf_dict[0][cid]["OP2"][QoI].dropna().iloc[-1])
        for cid in range(len(lf_dict[0]))]
)
# CoIs
CoI = sm_config["optim"]["CoI"]
lf_a_ADP = np.array([lf_dict[0][cid]["ADP"][CoI].dropna().iloc[-1] for cid in range(len(lf_dict[0]))])
lf_a_OP1 = np.array([lf_dict[0][cid]["OP1"][CoI].dropna().iloc[-1] for cid in range(len(lf_dict[0]))])
lf_a_OP2 = np.array([lf_dict[0][cid]["OP2"][CoI].dropna().iloc[-1] for cid in range(len(lf_dict[0]))])

# hf_DOE
# QoIs
hf_w_ADP = np.array([hf_dict[0][cid]["ADP"][QoI].iloc[-1] for cid in range(len(hf_dict[0]))])
hf_w_OP = np.array(
    [0.5 * (hf_dict[0][cid]["OP1"][QoI].dropna().iloc[-1] + hf_dict[0][cid]["OP2"][QoI].dropna().iloc[-1])
        for cid in range(len(hf_dict[0]))]
)
# CoIs
hf_a_ADP = np.array([hf_dict[0][cid]["ADP"][CoI].dropna().iloc[-1] for cid in range(len(hf_dict[0]))])
hf_a_OP1 = np.array([hf_dict[0][cid]["OP1"][CoI].dropna().iloc[-1] for cid in range(len(hf_dict[0]))])
hf_a_OP2 = np.array([hf_dict[0][cid]["OP2"][CoI].dropna().iloc[-1] for cid in range(len(hf_dict[0]))])

In [8]:
x_lf = np.vstack([x_lf, np.zeros(x_lf.shape[-1])])
lf_w_ADP = np.append(lf_w_ADP, sm_config["optim"]["bsl_lf_w_ADP"])
lf_w_OP = np.append(lf_w_OP, sm_config["optim"]["bsl_lf_w_OP"])
# CoI
lf_a_ADP = np.append(lf_a_ADP, sm_config["optim"]["bsl_lf_a_ADP"])
lf_a_OP1 = np.append(lf_a_OP1, sm_config["optim"]["bsl_lf_a_OP1"])
lf_a_OP2 = np.append(lf_a_OP2, sm_config["optim"]["bsl_lf_a_OP2"])
# addition of hf baseline results
# QoI
x_hf = np.vstack([x_hf, np.zeros(x_hf.shape[-1])])
hf_w_ADP = np.append(hf_w_ADP, sm_config["optim"]["bsl_hf_w_ADP"])
hf_w_OP = np.append(hf_w_OP, sm_config["optim"]["bsl_hf_w_OP"])
# CoI
hf_a_ADP = np.append(hf_a_ADP, sm_config["optim"]["bsl_hf_a_ADP"])
hf_a_OP1 = np.append(hf_a_OP1, sm_config["optim"]["bsl_hf_a_OP1"])
hf_a_OP2 = np.append(hf_a_OP2, sm_config["optim"]["bsl_hf_a_OP2"])

In [9]:
set_mo_DOE(model_loss, x_lf=x_lf, y_lf=[lf_w_ADP, lf_w_OP], x_hf=x_hf, y_hf=[hf_w_ADP, hf_w_OP])
model_loss.train()

In [10]:
hf_w_ADP

array([0.04231589, 0.04123692, 0.04450975, 0.04212306, 0.04382375,
       0.05779699, 0.03275494, 0.03127165, 0.05379593, 0.0349    ])

In [11]:
hf_w_OP

array([0.07005788, 0.06077774, 0.06271212, 0.05153917, 0.05782643,
       0.0669216 , 0.05467749, 0.05652358, 0.05891811, 0.0483    ])

In [12]:
lf_w_ADP

array([0.06228944, 0.07757496, 0.22242531, 0.1675026 , 0.08193779,
       0.06994403, 0.09878104, 0.08953398, 0.07194069, 0.05849092,
       0.09112923, 0.07035543, 0.0655037 , 0.11298419, 0.11326132,
       0.06431406, 0.23986814, 0.16233601, 0.09195042, 0.07204038,
       0.07756592, 0.13619896, 0.10946445, 0.0600521 , 0.09119289,
       0.06931943, 0.11633091, 0.0683741 , 0.05888479, 0.09883568,
       0.08184602, 0.05693617, 0.10886139, 0.05252851, 0.07724773,
       0.23174787, 0.25371779, 0.05222858, 0.22668785, 0.04952912,
       0.06051544, 0.09845866, 0.06619095, 0.12336388, 0.09402368,
       0.09856459, 0.06148682, 0.21012162, 0.14143413, 0.0765189 ,
       0.1547023 , 0.1034513 , 0.09384698, 0.20861503, 0.07566027,
       0.17624412, 0.08980026, 0.10124489, 0.10802921, 0.0586817 ,
       0.16359789, 0.05857248, 0.08408609, 0.0927076 , 0.06419824,
       0.0730836 , 0.07615512, 0.06956636, 0.07245694, 0.08083774,
       0.08933144, 0.06039337, 0.24104229, 0.23133323, 0.08576

In [13]:
hf_a_ADP

array([-3.45264165,  0.8535252 ,  6.62291413, -2.21939385, -4.39426583,
        5.12662958, -3.42821054,  4.70114174, -4.40037868,  0.131     ])